# 11b — Dipole Scatter Plot: psfFlux vs (scienceFlux − templateFlux)

This notebook produces two diagnostic scatter-plot grids (2 × 3 subplots each)
using **all stellar categories merged together**.

**Physical motivation:**
- `scienceFlux` is the pre-subtraction (coadd-level) stellar flux  ≥ 0.
- `templateFlux` is the reference-coadd flux  ≥ 0.
- `diff_flux = scienceFlux − templateFlux` captures the visit-to-template flux change.
- `psfFlux` is the PSF-fit flux measured directly on the difference image.
- For a perfect subtraction `psfFlux ≈ diff_flux`.
- Dipole-flagged sources deviate from this relation.

**Figure A — 2 × 3 scatter: Y = psfFlux, X = scienceFlux − templateFlux**
- All categories overplotted (different marker colours per category)
- Marker style: open circle (○) = dipole; filled disk (●) = non-dipole
- Marker colour indexed on AB magnitude of scienceFlux (colorbar per subplot)
- X and Y axis limits derived from data percentiles (p1–p99) — positive range
- Reference line Y = X shown

**Figure B — 2 × 3 scatter: Y = psfFlux / (scienceFlux − templateFlux), X = scienceFlux − templateFlux**
- Same marker convention
- Ratio Y = 1 reference line shown
- Y axis clipped to a sensible range (robust percentiles)

**Figure C — 2 × 3 scatter: Y = psfFlux / (scienceFlux − templateFlux), X = AB mag(scienceFlux)**
- Marker colour = `dipoleMeanFlux` amplitude (cmap jet: red = strong, blue = weak)
- Grey markers for sources without a dipole fit
- Ratio Y = 1 reference line; X axis inverted (bright left)

**Stellar categories:**
- `gaia_star_stable_hq` — Gaia stable HQ (primary calibration targets)
- `gaia_nophotgstar_stable_unknown_parallax` — Gaia stable, no photometric solution
- `gaia_star_variable` — Gaia variable stars (control group)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS, Université Paris-Saclay)  
**Creation Date:** 2026-05-16  
**Last update:** 2026-05-16 — added Figure C (ratio vs AB mag, colour = dipole amplitude)  
**Notebook tag:** 11b

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

print(f"pandas    : {pd.__version__}")
print(f"numpy     : {np.__version__}")
print(f"matplotlib: {mpl.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl not found → inline backend")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input parquets from notebook 01
# DIR_FIGS = f"figs_{NB_TAG}_11b"  # shared output directory with notebook 11
DIR_FIGS = "figs_FINK_BLOCK_LC_11b"
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands — display order: row 0 = u g r, row 1 = i z y ───────────
BANDS = list("ugrizy")
BANDS_ROW0 = list("ugr")
BANDS_ROW1 = list("izy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Stellar categories ─────────────────────────────────────────────────────────
CATEGORIES = {
    "gaia_star_stable_hq": {
        "label": "Gaia stable HQ",
        "color": "steelblue",
    },
    "gaia_nophotgstar_stable_unknown_parallax": {
        "label": "Gaia stable no-phot",
        "color": "seagreen",
    },
    "gaia_star_variable": {
        "label": "Gaia variable",
        "color": "firebrick",
    },
}

# ── AB magnitude calibration ───────────────────────────────────────────────────
AB_FLUX_ZERO = 3631e9  # nJy  (m_AB = -2.5 log10(f_nJy / AB_FLUX_ZERO))

# ── Colormap for marker colour (AB mag of scienceFlux) ────────────────────────
CMAP_NAME = "viridis_r"  # bright → yellow ; faint → purple
MAG_VMIN = 15.0  # bright limit (AB mag)
MAG_VMAX = 27.0  # faint  limit (AB mag)

# ── Marker sizes & transparencies ─────────────────────────────────────────────
MS_NODIP = 18  # filled disk — non-dipole
MS_DIP = 45  # open circle — dipole
ALPHA_NODIP = 0.30
ALPHA_DIP = 0.85

# ── Axis-limit percentiles (applied independently to X and Y) ─────────────────
# Values between 0 and 100; widen if data are clipped too aggressively.
PLOW = 1.0  # lower percentile for axis minimum
PHIGH = 99.0  # upper percentile for axis maximum
AXIS_MARGIN = 0.05  # fractional padding beyond [p_low, p_high]

# ── Ratio plot Y-axis clip percentiles ────────────────────────────────────────
RATIO_PLOW = 2.0
RATIO_PHIGH = 98.0

RATIO_YMAX = 1.5
RATIO_YMIN = 0.5

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save the current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data dir : {os.path.abspath(DIR_DATA)}")
print(f"Figs dir : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_nJy_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude.  Returns NaN for non-positive flux."""
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(f > 0, -2.5 * np.log10(f / AB_FLUX_ZERO), np.nan)


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Coerce a dipole-flag column (bool / int / str) to a boolean Series."""

    def _cast(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


def robust_lim(arr, plow=PLOW, phigh=PHIGH, margin=AXIS_MARGIN):
    """
    Return (vmin, vmax) computed from percentiles of arr,
    with a fractional margin added on each side.
    Falls back to (0, 1) if arr contains no finite values.
    """
    finite = arr[np.isfinite(arr)]
    if len(finite) < 2:
        return 0.0, 1.0
    vlo, vhi = np.percentile(finite, [plow, phigh])
    span = vhi - vlo
    if span == 0:
        span = abs(vhi) if vhi != 0 else 1.0
    return vlo - margin * span, vhi + margin * span


print("Utility functions defined.")

## 3. Load and merge parquet data

All three stellar categories are loaded and concatenated into a **single DataFrame**
with a `gaia_origin` column for identification.  Both figures will overplot all
categories, colour-coded by `mag_science` (AB mag of `scienceFlux`).

The column `r:dipoleMeanFlux` is loaded alongside the flux columns so that
Figure C (Section 9) can use it directly without a second pass over the parquets.

**Sign conventions:**
- `scienceFlux`, `templateFlux` : ≥ 0 (stellar flux in nJy)
- `diff_flux = scienceFlux − templateFlux` : can be slightly negative for
  constant stars due to noise, but is positive on average for brightening sources
- `psfFlux` : PSF-fit flux on the difference image — can be positive or negative,
  but for most stellar alerts it is positive and comparable to `diff_flux`
- `dipoleMeanFlux` : mean absolute flux of the two dipole lobes (nJy) — proxy for
  dipole artefact strength; `dipoleFluxDiff` is identically 0 (symmetric model)

In [ ]:
frames = []  # list of per-category DataFrames

for cat in CATEGORIES:
    fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: file not found ({fpath})")
        continue

    df = pd.read_parquet(fpath)

    # ── Mandatory columns ──────────────────────────────────────────────────
    required = ["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr", "r:band"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[WARN] {cat}: missing columns {missing} — skipped")
        continue

    df = df.dropna(subset=["r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

    # ── Dipole flag ────────────────────────────────────────────────────────
    if "r:isDipole" in df.columns:
        df["is_dipole"] = parse_dipole_bool(df["r:isDipole"].fillna(False))
    else:
        print(f"[WARN] {cat}: no r:isDipole column — all set to False")
        df["is_dipole"] = False

    # ── scienceFlux ────────────────────────────────────────────────────────
    if "r:scienceFlux" in df.columns:
        df["scienceFlux"] = df["r:scienceFlux"].astype(float)
        df["scienceFluxErr"] = (
            df["r:scienceFluxErr"].astype(float) if "r:scienceFluxErr" in df.columns else 0.0
        )
    else:
        print(f"[INFO] {cat}: r:scienceFlux absent — using |psfFlux| as proxy")
        df["scienceFlux"] = np.abs(df["r:psfFlux"].values)
        df["scienceFluxErr"] = df["r:psfFluxErr"].values

    # ── templateFlux ───────────────────────────────────────────────────────
    if "r:templateFlux" in df.columns:
        df["templateFlux"] = df["r:templateFlux"].astype(float)
        df["templateFluxErr"] = (
            df["r:templateFluxErr"].astype(float) if "r:templateFluxErr" in df.columns else 0.0
        )
    else:
        print(f"[INFO] {cat}: r:templateFlux absent — template terms set to 0")
        df["templateFlux"] = 0.0
        df["templateFluxErr"] = 0.0

    # ── Derived columns ────────────────────────────────────────────────────
    df["psfFlux"] = df["r:psfFlux"].astype(float)
    df["psfFluxErr"] = df["r:psfFluxErr"].astype(float)

    df["diff_flux"] = df["scienceFlux"] - df["templateFlux"]  # X axis
    df["diff_fluxErr"] = np.sqrt(df["scienceFluxErr"] ** 2 + df["templateFluxErr"] ** 2)

    # Ratio Y = psfFlux / diff_flux  (Figure B and Figure C)
    with np.errstate(invalid="ignore", divide="ignore"):
        df["ratio"] = np.where(
            np.abs(df["diff_flux"]) > 0,
            df["psfFlux"] / df["diff_flux"],
            np.nan,
        )
        # Error propagation: σ_ratio = |ratio| * sqrt((σ_psf/psfFlux)² + (σ_diff/diff_flux)²)
        df["ratioErr"] = np.where(
            (np.abs(df["psfFlux"]) > 0) & (np.abs(df["diff_flux"]) > 0),
            np.abs(df["ratio"])
            * np.sqrt((df["psfFluxErr"] / df["psfFlux"]) ** 2 + (df["diff_fluxErr"] / df["diff_flux"]) ** 2),
            np.nan,
        )

    # AB magnitude of scienceFlux — used for marker colour in Fig A & B
    df["mag_science"] = flux_nJy_to_mag_AB(df["scienceFlux"].values)

    # Category provenance
    df["gaia_origin"] = cat
    df["cat_label"] = CATEGORIES[cat]["label"]

    frames.append(df)
    lbl = CATEGORIES[cat]["label"]
    n_tot = len(df)
    n_dip = int(df["is_dipole"].sum())
    print(f"  {lbl:40s}  {n_tot:6d} rows  {n_dip:5d} dipoles  ({100 * n_dip / n_tot:.1f}%)")

# ── Concatenate all categories ─────────────────────────────────────────────────
if not frames:
    raise RuntimeError("No parquet files loaded — check DIR_DATA path.")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows (all categories): {len(df_all):,}")
print(
    f"Total dipoles             : {int(df_all['is_dipole'].sum()):,}  "
    f"({100 * df_all['is_dipole'].mean():.1f}%)"
)

## 4. Quick inspection

In [ ]:
cols_show = [
    "gaia_origin",
    "r:band",
    "psfFlux",
    "psfFluxErr",
    "scienceFlux",
    "templateFlux",
    "diff_flux",
    "diff_fluxErr",
    "ratio",
    "ratioErr",
    "mag_science",
    "is_dipole",
]
cols_present = [c for c in cols_show if c in df_all.columns]
print("Preview (first 8 rows of merged table):")
display(df_all[cols_present].head(8))

print("\nPer-band source count:")
display(
    df_all.groupby("r:band")[["is_dipole"]]
    .agg(["count", "sum"])
    .rename(columns={"count": "N_total", "sum": "N_dipole"})
)

## 5. Figure A — psfFlux vs (scienceFlux − templateFlux)

**Axes:**
- X : `diff_flux = scienceFlux − templateFlux`  (nJy)
- Y : `psfFlux`  (nJy)

**Axis limits:** derived independently for X and Y from the p1–p99 percentile
range of the band data.  Both fluxes are generally positive for stellar sources;
the limits therefore reflect the actual data range rather than a symmetric window.

**Marker convention:**
- Filled disk (●) : not a dipole — colour = AB mag of scienceFlux
- Open circle (○) : dipole flagged — edge colour = AB mag of scienceFlux

**Reference line:** Y = X (ideal equality between the two flux estimators).

In [ ]:
def _scatter_one_band(ax, df_b, cmap, norm, xcol, ycol, xerrcol, yerrcol, mag_vmin, mag_vmax):
    """
    Draw non-dipole (filled disk) and dipole (open circle) scatter points
    on axes `ax` for the rows in df_b.
    Returns (n_nodip, n_dip).
    """
    df_b = df_b.dropna(subset=[xcol, ycol, "mag_science"])
    df_nodip = df_b[~df_b["is_dipole"]]
    df_dip = df_b[df_b["is_dipole"]]
    n_nodip = len(df_nodip)
    n_dip = len(df_dip)

    def mag_to_rgba(mag_series):
        clipped = np.clip(mag_series.values, mag_vmin, mag_vmax)
        return cmap(norm(clipped))

    # Non-dipole: filled disks
    if n_nodip > 0:
        rgba_nd = mag_to_rgba(df_nodip["mag_science"])
        ax.errorbar(
            df_nodip[xcol].values,
            df_nodip[ycol].values,
            xerr=df_nodip[xerrcol].values if xerrcol else None,
            yerr=df_nodip[yerrcol].values if yerrcol else None,
            fmt="none",
            ecolor="silver",
            elinewidth=0.4,
            alpha=0.35,
            zorder=1,
        )
        ax.scatter(
            df_nodip[xcol].values,
            df_nodip[ycol].values,
            s=MS_NODIP,
            c=rgba_nd,
            marker="o",
            alpha=ALPHA_NODIP,
            linewidths=0,
            zorder=2,
            label=f"non-dipole (N={n_nodip})",
        )

    # Dipole: open circles
    if n_dip > 0:
        rgba_dip = mag_to_rgba(df_dip["mag_science"])
        ax.errorbar(
            df_dip[xcol].values,
            df_dip[ycol].values,
            xerr=df_dip[xerrcol].values if xerrcol else None,
            yerr=df_dip[yerrcol].values if yerrcol else None,
            fmt="none",
            ecolor="tomato",
            elinewidth=0.7,
            alpha=0.65,
            zorder=3,
        )
        ax.scatter(
            df_dip[xcol].values,
            df_dip[ycol].values,
            s=MS_DIP,
            c="none",
            edgecolors=rgba_dip,
            marker="o",
            linewidths=1.5,
            alpha=ALPHA_DIP,
            zorder=4,
            label=f"dipole (N={n_dip})",
        )

    return n_nodip, n_dip


print("_scatter_one_band() helper defined.")

In [ ]:
def plot_figA(df, savename=None, cmap_name=CMAP_NAME, mag_vmin=MAG_VMIN, mag_vmax=MAG_VMAX):
    """
    Figure A: Y = psfFlux   vs   X = scienceFlux − templateFlux  (all categories merged).

    Axis limits are computed from p1–p99 of the per-band data (positive range).
    Reference line Y = X is drawn.
    """
    cmap = cm.get_cmap(cmap_name)
    norm = mpl.colors.Normalize(vmin=mag_vmin, vmax=mag_vmax)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 5.0 * 2), squeeze=False)
    fig.suptitle(
        "All categories — psfFlux  vs  scienceFlux $-$ templateFlux\n"
        "● non-dipole   ○ dipole   colour = scienceFlux AB mag",
        fontsize=12,
        fontweight="bold",
        y=1.01,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            df_b = df[df["r:band"] == band].copy()

            n_nodip, n_dip = _scatter_one_band(
                ax,
                df_b,
                cmap,
                norm,
                xcol="diff_flux",
                ycol="psfFlux",
                xerrcol="diff_fluxErr",
                yerrcol="psfFluxErr",
                mag_vmin=mag_vmin,
                mag_vmax=mag_vmax,
            )

            # ── Axis limits: robust percentile of the band data ───────────
            xdata = df_b["diff_flux"].dropna().values
            ydata = df_b["psfFlux"].dropna().values
            xmin, xmax = robust_lim(xdata)
            ymin, ymax = robust_lim(ydata)
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

            # ── Reference line Y = X ──────────────────────────────────────
            ref_lo = max(xmin, ymin)
            ref_hi = min(xmax, ymax)
            if ref_lo < ref_hi:
                x_ref = np.linspace(ref_lo, ref_hi, 200)
                ax.plot(x_ref, x_ref, "k--", lw=1.0, alpha=0.55, label="Y = X", zorder=0)

            # Horizontal & vertical reference at 0
            ax.axhline(0, color="grey", lw=0.6, ls=":", alpha=0.5)
            ax.axvline(0, color="grey", lw=0.6, ls=":", alpha=0.5)

            # ── Colorbar ──────────────────────────────────────────────────
            cb = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
            cb.set_label("scienceFlux (AB mag)", fontsize=7)
            cb.ax.tick_params(labelsize=6)
            cb.ax.invert_yaxis()  # bright (low mag) at bottom

            # ── Labels ────────────────────────────────────────────────────
            ax.set_title(
                f"band  {band}   [{n_nodip + n_dip} sources  /  {n_dip} dipoles]",
                color=bcolor,
                fontweight="bold",
                fontsize=10,
            )
            ax.set_xlabel(r"scienceFlux $-$ templateFlux  (nJy)", fontsize=9)
            ax.set_ylabel(r"psfFlux  (nJy)", fontsize=9)
            leg = ax.legend(fontsize=7, loc="upper left", framealpha=0.7)
            for lh in leg.legend_handles:
                lh.set_alpha(0.9)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_figA() defined.")

In [ ]:
plot_figA(df_all, cmap_name="jet_r", savename="11b_figA_psfFlux_vs_diff")

## 6. Figure B — psfFlux / (scienceFlux − templateFlux) vs (scienceFlux − templateFlux)

**Axes:**
- X : `diff_flux = scienceFlux − templateFlux`  (nJy)
- Y : `ratio = psfFlux / diff_flux`  (dimensionless)

**Physical meaning of the ratio:**
- `ratio = 1` → perfect agreement between the PSF-fit flux and the pixel-sum
  flux difference: the subtraction is unbiased.
- `ratio > 1` → psfFlux overestimates the difference: possible dipole artefact
  inflating the residual.
- `ratio < 1` → psfFlux underestimates the change: template over-subtraction
  or centroid offset.
- Rows with `diff_flux ≤ 0` (noise-dominated or slightly over-subtracted) are
  excluded from this plot (ratio undefined or ill-conditioned).

**Y axis limits:** derived from p2–p98 of the per-band ratio data to
exclude extreme outliers while preserving the bulk distribution.

**X axis limits:** same as Figure A (p1–p99 of diff_flux).

In [ ]:
def plot_figB(
    df,
    savename=None,
    cmap_name=CMAP_NAME,
    mag_vmin=MAG_VMIN,
    mag_vmax=MAG_VMAX,
    YMIN=RATIO_YMIN,
    YMAX=RATIO_YMAX,
):
    """
    Figure B: Y = psfFlux / (scienceFlux − templateFlux)
              X = scienceFlux − templateFlux   (all categories merged).

    Only rows with diff_flux > 0 are plotted (ratio is well-defined).
    Y-axis limits are derived from p2–p98 of the ratio to suppress outliers.
    Reference line Y = 1 is drawn.
    """
    cmap = cm.get_cmap(cmap_name)
    norm = mpl.colors.Normalize(vmin=mag_vmin, vmax=mag_vmax)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 5.0 * 2), squeeze=False)
    fig.suptitle(
        "All categories — psfFlux / (scienceFlux $-$ templateFlux)  "
        "vs  scienceFlux $-$ templateFlux\n"
        "● non-dipole   ○ dipole   colour = scienceFlux AB mag   "
        "(only diff_flux > 0 shown)",
        fontsize=14,
        fontweight="bold",
        y=1.0,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            # Keep only rows with positive diff_flux (ratio is meaningful)
            df_b = df[(df["r:band"] == band) & (df["diff_flux"] > 0)].copy()

            n_nodip, n_dip = _scatter_one_band(
                ax,
                df_b,
                cmap,
                norm,
                xcol="diff_flux",
                ycol="ratio",
                xerrcol="diff_fluxErr",
                yerrcol="ratioErr",
                mag_vmin=mag_vmin,
                mag_vmax=mag_vmax,
            )

            # ── Axis limits ───────────────────────────────────────────────
            xdata = df_b["diff_flux"].dropna().values
            ydata = df_b["ratio"].dropna().values
            xmin, xmax = robust_lim(xdata, plow=PLOW, phigh=PHIGH)
            ymin, ymax = robust_lim(ydata, plow=RATIO_PLOW, phigh=RATIO_PHIGH)
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

            # ── Reference line Y = 1 ──────────────────────────────────────
            ax.axhline(1.0, color="black", lw=1.0, ls="--", alpha=0.6, label="Y = 1", zorder=0)
            ax.axhline(0, color="grey", lw=0.6, ls=":", alpha=0.4)
            ax.axvline(0, color="grey", lw=0.6, ls=":", alpha=0.4)

            # ── Colorbar ──────────────────────────────────────────────────
            cb = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
            cb.set_label("scienceFlux (AB mag)", fontsize=10)
            cb.ax.tick_params(labelsize=10)
            cb.ax.invert_yaxis()

            # ── Labels ────────────────────────────────────────────────────
            ax.set_title(
                f"band  {band}   [{n_nodip + n_dip} sources  /  {n_dip} dipoles]",
                color=bcolor,
                fontweight="bold",
                fontsize=14,
            )
            ax.set_xlabel(r"scienceFlux $-$ templateFlux  (nJy)", fontsize=10)
            ax.set_ylabel(
                r"psfFlux / (scienceFlux $-$ templateFlux)",
                fontsize=10,
            )
            ax.set_ylim(YMIN, YMAX)
            leg = ax.legend(fontsize=10, loc="upper right", framealpha=0.7)
            for lh in leg.legend_handles:
                lh.set_alpha(0.9)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_figB() defined.")

In [ ]:
plot_figB(df_all, cmap_name="jet_r", savename="11b_figB_ratio_vs_diff")

## 7. Per-category breakdown: flux statistics

Median and MAD of `psfFlux`, `diff_flux`, and their ratio,
split by category × band × dipole flag.
Useful to check whether one category dominates the scatter.

In [ ]:
def mad(x):
    """Median absolute deviation."""
    m = np.nanmedian(x)
    return np.nanmedian(np.abs(x - m))


rows_stat = []
for cat in df_all["gaia_origin"].unique():
    lbl = CATEGORIES[cat]["label"]
    sub_cat = df_all[df_all["gaia_origin"] == cat]
    for band in BANDS:
        for is_dip, tag in [(False, "non-dipole"), (True, "dipole")]:
            sub = sub_cat[(sub_cat["r:band"] == band) & (sub_cat["is_dipole"] == is_dip)]
            if len(sub) == 0:
                continue
            rows_stat.append(
                {
                    "category": lbl,
                    "band": band,
                    "type": tag,
                    "N": len(sub),
                    "med_psfFlux": np.nanmedian(sub["psfFlux"].values),
                    "mad_psfFlux": mad(sub["psfFlux"].values),
                    "med_diff": np.nanmedian(sub["diff_flux"].values),
                    "mad_diff": mad(sub["diff_flux"].values),
                    "med_ratio": np.nanmedian(sub["ratio"].values),
                }
            )

df_stat = pd.DataFrame(rows_stat)
print("Flux statistics per category × band × dipole flag (nJy):")
display(
    df_stat.style.format(
        {
            "med_psfFlux": "{:.1f}",
            "mad_psfFlux": "{:.1f}",
            "med_diff": "{:.1f}",
            "mad_diff": "{:.1f}",
            "med_ratio": "{:.3f}",
        }
    )
)

## 8. Conclusions (Figures A & B)

**Figure A** (`psfFlux` vs `diff_flux`):
- Points on the Y = X identity line indicate a consistent flux estimator.
- Dipole open circles (○) displaced **above** Y = X mean the PSF-fit flux is
  larger than the naive pixel-sum difference → classic dipole inflation.
- Dipole points clustered near the X-axis (low psfFlux, finite diff_flux) would
  suggest template over-subtraction independently of the PSF fit.

**Figure B** (`ratio` vs `diff_flux`):
- Ratio = 1 → unbiased subtraction.
- Dipoles with ratio ≫ 1 at small `diff_flux` values signal artefacts where
  the PSF-fit picks up large residuals even though the integrated pixel difference
  is small — the hallmark of a spatial dipole morphology.
- A band-dependent trend in the ratio baseline (e.g. systematic ratio > 1 in u-band)
  would point to PSF-model errors in that filter.

> **Recommended next step:** identify the objects with the largest ratio values
> in Figure B (especially dipole-flagged) and fetch their Butler difference-image
> stamps via notebook 12b to confirm the dipole morphology visually.

## 9. Figure C — psfFlux / (scienceFlux − templateFlux) vs AB mag(scienceFlux)

**Axes:**
- X : AB magnitude of `scienceFlux`  (mag, bright left → faint right, axis inverted)
- Y : `ratio = psfFlux / (scienceFlux − templateFlux)`  (dimensionless)

**Marker colour:** amplitude of the dipole = `r:dipoleMeanFlux` (nJy),  
mapped with **cmap `jet`** — **red = strong dipole, blue = weak dipole**.  
The colour normalisation is **logarithmic** and **global** across all bands
(p2–p98 of `dipoleMeanFlux` over all rows), so colours are directly comparable
between subplots.  Sources without a dipole amplitude fit are plotted in light grey.

**Marker style:**
- Light grey filled disk      : no dipole amplitude (`dipoleMeanFlux` NaN)
- Colour disk, grey edge      : has dipole amplitude but `isDipole=False`
- Colour disk, **black** edge : `isDipole=True` (AP dipole-fit accepted)

**Error bars:**
- X : σ(mag) = 2.5/ln(10) × σ(scienceFlux)/scienceFlux
- Y : σ(ratio) propagated from σ(psfFlux) and σ(diff_flux)

**Purpose:** check whether the flux-ratio bias depends on stellar brightness
and whether dipole amplitude (`dipoleMeanFlux`) correlates with the deviation
from ratio = 1.

> **Note on dipole amplitude column:**  
> `r:dipoleMeanFlux` is the mean absolute flux of the two dipole lobes as fitted
> by the AP dipole model (nJy); it scales with PSF peak flux and is a proxy for
> the *strength* of the subtraction artefact.  
> `r:dipoleFluxDiff` is identically 0 in these data (symmetric dipole assumption)
> and is **not used** here.

In [ ]:
# ── Figure C parameters ────────────────────────────────────────────────────────
CMAP_DIPOLE = "jet"  # red = strong dipole, blue = weak dipole
DIPOLE_AMP_COL = "r:dipoleMeanFlux"  # column carrying dipole amplitude (nJy)
DIPOLE_AMP_PLOW = 2.0  # percentile for cmap lower bound
DIPOLE_AMP_PHIGH = 98.0  # percentile for cmap upper bound

# Y-axis clip percentiles for the ratio
FIGC_RATIO_PLOW = 2.0
FIGC_RATIO_PHIGH = 98.0

# X-axis: magnitude range
FIGC_MAG_PLOW = 1.0
FIGC_MAG_PHIGH = 99.0

# Marker sizes
FIGC_MS_NODIP = 18
FIGC_MS_DIP = 40


def _ensure_dipole_amp(df_in):
    """Attach r:dipoleMeanFlux to df_in if not already present.

    The column is loaded from each category parquet (join on r:diaSourceId)
    if it was not carried through by the Section-3 merge.
    """
    if DIPOLE_AMP_COL in df_in.columns:
        n_ok = df_in[DIPOLE_AMP_COL].notna().sum()
        print(f"{DIPOLE_AMP_COL} already present — {n_ok} non-NaN values.")
        return df_in

    print(f"{DIPOLE_AMP_COL} missing — reloading from parquets to attach it…")
    amp_frames = []
    for cat in CATEGORIES:
        fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
        if not os.path.exists(fpath):
            continue
        tmp = pd.read_parquet(fpath, columns=["r:diaSourceId", DIPOLE_AMP_COL])
        amp_frames.append(tmp)
    if not amp_frames:
        print("  No parquet files found — dipoleMeanFlux will be NaN.")
        df_in[DIPOLE_AMP_COL] = np.nan
        return df_in
    amp_df = pd.concat(amp_frames, ignore_index=True).drop_duplicates("r:diaSourceId")
    df_in = df_in.merge(amp_df, on="r:diaSourceId", how="left")
    n_ok = df_in[DIPOLE_AMP_COL].notna().sum()
    print(f"  Attached {DIPOLE_AMP_COL}: {n_ok} / {len(df_in)} non-NaN values.")
    return df_in


df_all = _ensure_dipole_amp(df_all)


def plot_figC(df, savename=None):
    """
    Figure C: Y = psfFlux / (scienceFlux − templateFlux)
              X = AB magnitude of scienceFlux  (axis inverted: bright left)
              Colour = dipoleMeanFlux (nJy), LogNorm, cmap jet.
                       red = strong dipole  |  blue = weak dipole

    Layout  : 2 × 3 subplots (u g r top row / i z y bottom row).
    Filter  : only rows with diff_flux > 0 and finite mag_science plotted.
    Colour  : global LogNorm derived from p2–p98 of dipoleMeanFlux across all bands
              (consistent colour scale between subplots).
    """
    cmap_dip = plt.get_cmap(CMAP_DIPOLE)

    # ── Global colour normalisation from dipoleMeanFlux ───────────────────
    amp_all = df[DIPOLE_AMP_COL].dropna().values.astype(float)
    amp_all = amp_all[np.isfinite(amp_all) & (amp_all > 0)]
    if len(amp_all) >= 2:
        amp_vmin = float(np.percentile(amp_all, DIPOLE_AMP_PLOW))
        amp_vmax = float(np.percentile(amp_all, DIPOLE_AMP_PHIGH))
    else:
        amp_vmin, amp_vmax = 1.0, 1e6
    norm_dip = mpl.colors.LogNorm(vmin=max(amp_vmin, 1.0), vmax=amp_vmax)
    sm_dip = mpl.cm.ScalarMappable(cmap=cmap_dip, norm=norm_dip)
    sm_dip.set_array([])
    print(f"dipoleMeanFlux colour scale (log): [{amp_vmin:.0f}, {amp_vmax:.0f}] nJy")

    fig, axes = plt.subplots(2, 3, figsize=(5.5 * 3, 5.0 * 2), squeeze=False)
    fig.suptitle(
        r"psfFlux / (scienceFlux $-$ templateFlux)   vs   AB mag(scienceFlux)"
        "\n"
        r"Colour = dipoleMeanFlux (nJy, log scale) — "
        "cmap jet:  red = strong dipole,  blue = weak dipole  |  grey = no dipole fit",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            # Keep only rows with positive diff_flux and finite mag_science / ratio
            df_b = df[
                (df["r:band"] == band)
                & (df["diff_flux"] > 0)
                & np.isfinite(df["mag_science"].values.astype(float))
                & np.isfinite(df["ratio"].values.astype(float))
            ].copy()

            if df_b.empty:
                ax.set_title(f"band {band} — no data", color=bcolor)
                ax.axis("off")
                continue

            # ── Axis limits ───────────────────────────────────────────────
            x_arr = df_b["mag_science"].values.astype(float)
            y_arr = df_b["ratio"].values.astype(float)
            xmin, xmax = robust_lim(x_arr, plow=FIGC_MAG_PLOW, phigh=FIGC_MAG_PHIGH)
            ymin, ymax = robust_lim(y_arr, plow=FIGC_RATIO_PLOW, phigh=FIGC_RATIO_PHIGH)
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymin, ymax)

            # ── Masks: with / without dipole amplitude ────────────────────
            amp_vals = df_b[DIPOLE_AMP_COL].values.astype(float)
            has_amp = np.isfinite(amp_vals) & (amp_vals > 0)
            df_noamp = df_b[~has_amp]
            df_amp = df_b[has_amp].copy()
            df_amp["_amp"] = amp_vals[has_amp]
            n_dip_flag = int(df_b["is_dipole"].sum())

            # ── Grey points: no dipole amplitude ─────────────────────────
            if len(df_noamp):
                ax.scatter(
                    df_noamp["mag_science"].values,
                    df_noamp["ratio"].values,
                    s=FIGC_MS_NODIP,
                    c="lightgrey",
                    edgecolors="none",
                    alpha=0.35,
                    linewidths=0,
                    zorder=2,
                    label=f"no dipole fit (N={len(df_noamp)})",
                )

            # ── Colour-mapped points: sources with dipoleMeanFlux ─────────
            if len(df_amp):
                amp_clipped = np.clip(df_amp["_amp"].values, norm_dip.vmin, norm_dip.vmax)
                colors = cmap_dip(norm_dip(amp_clipped))
                dip_mask = df_amp["is_dipole"].values.astype(bool)

                # Has amplitude but isDipole=False — grey edge
                if (~dip_mask).any():
                    ax.scatter(
                        df_amp.loc[~dip_mask, "mag_science"].values,
                        df_amp.loc[~dip_mask, "ratio"].values,
                        s=FIGC_MS_NODIP,
                        c=colors[~dip_mask],
                        edgecolors="grey",
                        linewidths=0.4,
                        alpha=0.55,
                        zorder=3,
                        label=f"has amp, isDipole=False (N={int((~dip_mask).sum())})",
                    )

                # isDipole=True — black edge, larger, front
                if dip_mask.any():
                    ax.scatter(
                        df_amp.loc[dip_mask, "mag_science"].values,
                        df_amp.loc[dip_mask, "ratio"].values,
                        s=FIGC_MS_DIP,
                        c=colors[dip_mask],
                        edgecolors="black",
                        linewidths=0.7,
                        alpha=0.85,
                        zorder=4,
                        label=f"isDipole=True (N={int(dip_mask.sum())})",
                    )

            # ── Error bars ────────────────────────────────────────────────
            if len(df_amp):
                with np.errstate(invalid="ignore", divide="ignore"):
                    sig_mag = np.where(
                        (df_amp["scienceFlux"].values > 0) & np.isfinite(df_amp["scienceFluxErr"].values),
                        2.5
                        / np.log(10)
                        * np.abs(df_amp["scienceFluxErr"].values / df_amp["scienceFlux"].values),
                        np.nan,
                    )
                ax.errorbar(
                    df_amp["mag_science"].values,
                    df_amp["ratio"].values,
                    xerr=sig_mag,
                    yerr=df_amp["ratioErr"].values,
                    fmt="none",
                    ecolor="#888888",
                    elinewidth=0.4,
                    capsize=0,
                    alpha=0.30,
                    zorder=1,
                )

            # ── Reference lines ───────────────────────────────────────────
            ax.axhline(1.0, color="black", lw=1.0, ls="--", alpha=0.55, zorder=0, label="ratio = 1")
            ax.axhline(0.0, color="grey", lw=0.5, ls=":", alpha=0.40, zorder=0)

            # ── Colorbar ──────────────────────────────────────────────────
            cb = plt.colorbar(sm_dip, ax=ax, fraction=0.046, pad=0.04)
            cb.set_label("dipoleMeanFlux (nJy)", fontsize=10)
            cb.ax.tick_params(labelsize=10)

            # ── Stats box ─────────────────────────────────────────────────
            n_tot_b = len(df_b)
            med_ratio = float(np.nanmedian(y_arr))
            ax.text(
                0.02,
                0.97,
                f"N={n_tot_b}  isDipole={n_dip_flag}\nmed(ratio)={med_ratio:.3f}",
                transform=ax.transAxes,
                fontsize=10,
                va="top",
                ha="left",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#aaa", alpha=0.85),
            )

            # ── Axis labels & legend ──────────────────────────────────────
            ax.set_title(
                f"band  {band}   [{n_tot_b} sources  /  {n_dip_flag} dipoles]",
                color=bcolor,
                fontweight="bold",
                fontsize=12,
            )
            ax.set_xlabel(r"AB mag(scienceFlux)", fontsize=10)
            ax.set_ylabel(
                r"psfFlux / (scienceFlux $-$ templateFlux)",
                fontsize=10,
            )
            ax.invert_xaxis()  # bright stars on the left
            ax.legend(
                fontsize=10,
                loc="upper right",
                framealpha=0.7,
                markerscale=1.2,
                handlelength=1.2,
            )
            ax.set_ylim(0.8, 1.2)
            ax.invert_xaxis()

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_figC() defined.")

In [ ]:
plot_figC(df_all, savename="11b_figC_ratio_vs_mag_dipoleAmp")

## 8. Conclusions

**Figure A** (`psfFlux` vs `diff_flux`):
- Points on the Y = X identity line indicate a consistent flux estimator.
- Dipole open circles (○) displaced **above** Y = X mean the PSF-fit flux is
  larger than the naive pixel-sum difference → classic dipole inflation.
- Dipole points clustered near the X-axis (low psfFlux, finite diff_flux) would
  suggest template over-subtraction independently of the PSF fit.

**Figure B** (`ratio` vs `diff_flux`):
- Ratio = 1 → unbiased subtraction.
- Dipoles with ratio ≫ 1 at small `diff_flux` values signal artefacts where
  the PSF-fit picks up large residuals even though the integrated pixel difference
  is small — the hallmark of a spatial dipole morphology.
- A band-dependent trend in the ratio baseline (e.g. systematic ratio > 1 in u-band)
  would point to PSF-model errors in that filter.

**Figure C** (`ratio` vs `mag_science`, colour = `dipoleMeanFlux`):
- Red markers (strong dipole amplitude) systematically far from ratio = 1 confirm
  that bright dipoles are the dominant source of flux-ratio bias.
- A trend of ratio deviating with stellar brightness would indicate a PSF-model
  or flux-scale problem that scales with source flux.
- Sources with large `dipoleMeanFlux` but `isDipole=False` (colour disk, grey edge)
  are candidates for which the dipole fit succeeded but the quality cut was not met.

> **Recommended next step:** identify the objects with the largest `dipoleMeanFlux`
> (reddest points in Figure C) and fetch their Butler difference-image stamps
> via notebook 12b to confirm the dipole morphology visually.

In [ ]:
print("Notebook 11b — dipole scatter plot — complete.")